# S1.5 — 메커니즘 경화 + S2 게이트 (a/b/c/d)

> **루프상 위치:** 가설 ✓ → S1 진단 ✓ → **S1.5(여기) = ③ 닫기 + S2 게이트** → S2.
>
> **우리 가설 = 3주장:**
> - ① 역전 (단기-best 층 ≠ 수렴-best 층) — **S1서 확정**
> - ② 단기 predictor (단기회복 ∝ ‖Hδ‖²) — **S1서 확정 (recipe 토대)**
> - ③ 수렴 설명 (수렴 ∝ δᵀHδ?) — **S1서 안 맞음 → 여기서 A/B 가른다**
>
> ①②는 ③과 *무관하게* 확정. S1.5는 ③의 원인(A 추정기 vs B 보상)을 가르고, S2(subset) 전제(가산성)를 점검한다.
>
> - **S1.5a** PSD model-Fisher(δᵀGδ) → δᵀHδ 실패가 추정기(A) 탓인지 본질(B)인지
> - **S1.5b** 3-anchor(FP vs PTQ 곡률) + cos(Δ,δ) → de-quant vs compensation (**보조** 증거)
> - **S1.5c** path-integrated → 동적 proxy가 예산의존 잡나
> - **S1.5d** 가산성 I_ij → 단일층 점수가 subset으로 전이되나 (S2 게이트)
>
> S1.2 산출물(recovery·proxies)을 *로드해 재활용* → 새 full-sweep 불필요. PSD proxy만 신규(학습 0, 몇 분).
> **판정엔 자동 임의컷 없음** — 표를 보고 사람이 판단, 5 seed서 확정.

In [ ]:
# --- Colab 셋업 ---
import os
REPO = '/content/26_Capstone'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/u-nsiq/26_Capstone.git {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(REPO)
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- 엔진 + Drive + 경로 (S1.5: S1.2 입력 로드 + s1_5 출력) ---
from qat_engine import *
import numpy as np, matplotlib.pyplot as plt, json as _json, csv as _csv, itertools

try:
    from google.colab import drive; drive.mount('/content/drive')
    ART = '/content/drive/MyDrive/26_Capstone'
except Exception:
    ART = './_local_art'
for sub in ['checkpoints','outputs','figures','outputs/s1_5','figures/s1_5']:
    os.makedirs(f'{ART}/{sub}', exist_ok=True)
DATA_ROOT = f'{ART}/data'
CKPT      = f'{ART}/checkpoints/resnet18_cifar100_fp32.pt'
S12DIR    = f'{ART}/outputs/s1_2'      # ← S1.2 recovery·proxies 로드
OUTDIR    = f'{ART}/outputs/s1_5'
FIG       = f'{ART}/figures/s1_5'
print('device', DEVICE, '| S1.2 in', S12DIR, '| S1.5 out', OUTDIR)

In [ ]:
# --- S1.5 설정 ---
BITS      = [4, 3, 2]
MAIN_BIT  = 3                 # 본 무대(recoverable)
SEEDS     = [0, 1, 2]         # 발표 전 3 (메커니즘/가산성 충분, 확정은 이후 5+)
SHORT, CONV   = 30, 2500
N_BATCH_PROXY = 8             # calib_size=1024/batch128 = 진짜 8배치 (정합)
PIG_T     = 30               # path-integrated step 수 (S1.5c; SHORT와 맞춤)
DIR_STEPS = 800              # cos(Δ,δ)용 단일층 QAT 예산 (S1.5b; 길수록 방향 깨끗·속도 trade = 표본선택)
ADD_TOPK  = 5                # 가산성 상위 K층 (S1.5d)
print('main bit', MAIN_BIT, '| seeds', SEEDS)

In [ ]:
# baseline + 층목록·비용(N) — calib_size=1024 → N_BATCH_PROXY=8 정합(Codex)
train_loader, val_loader, calib_loader = get_loaders('cifar100', batch=128, calib_size=1024, data_root=DATA_ROOT)
fp_model = load_model('resnet18', 'cifar100')
fp_model, fp_acc = train_baseline(fp_model, train_loader, val_loader, ckpt_path=CKPT, resume=True)
fp_model.eval()
_pm0   = make_ptq_model(fp_model, MAIN_BIT)
layers = list_quant_layers(_pm0)
costs  = get_layer_costs(_pm0, layers)
Nvec   = [costs[L] for L in layers]
print('fp_acc', round(fp_acc, 2), '| layers', len(layers))

## S1.5a — PSD model-Fisher (δᵀGδ) → A/B 분기

`δᵀHδ`가 수렴회복을 못 예측한 게 (A) 비-PSD 추정 탓인가, (B) 본질(보상)인가. PSD 보장 `δᵀGδ`로 재측정.

In [ ]:
# S1.5a-1: S1.2 recovery 로드 + PSD proxy(dtGd/normGd2) 신규 계산
def load_recovery_csv(path):
    R = {}
    with open(path) as f:
        rd = _csv.reader(f); header = next(rd)
        tcols = [c for c in header if c.startswith('t')]; ts = [int(c[1:]) for c in tcols]
        for row in rd:
            R[row[0]] = {ts[i]: float(row[2 + i]) for i in range(len(ts))}
    return R

REC = {}                       # REC[bit][layer][t] = mean loss-R (S1.2)
for bit in BITS:
    p = f'{S12DIR}/s1_2_recovery_loss_w{bit}.csv'
    if os.path.exists(p):
        REC[bit] = load_recovery_csv(p)
print('loaded S1.2 recovery:', list(REC.keys()))
assert MAIN_BIT in REC, f'S1.2 recovery_loss_w{MAIN_BIT}.csv 없음 — S1.2 먼저 돌렸는지/경로 확인'

PROX_PTQ = {}                  # anchor_proxies @ PTQ: {layer:{dtHd,normHd2,dtGd,normGd2}}
for bit in BITS:
    pm = make_ptq_model(fp_model, bit)
    deltas = quant_error(pm)
    PROX_PTQ[bit] = anchor_proxies(pm, deltas, calib_loader, layers, n_batches=N_BATCH_PROXY)
    _json.dump(PROX_PTQ[bit], open(f'{OUTDIR}/s1_5_psd_ptq_w{bit}.json', 'w'))
    neg  = sum(1 for L in layers if PROX_PTQ[bit][L]['dtHd'] < 0)
    negG = sum(1 for L in layers if PROX_PTQ[bit][L]['dtGd'] < 0)
    print(f'W{bit}: dtHd<0 {neg}/{len(layers)} 층(비-PSD) | dtGd<0 {negG}/{len(layers)} (PSD라 0이어야)')

In [ ]:
# S1.5a-2: A/B — δᵀHδ vs δᵀGδ(PSD)가 수렴회복 예측 + calib.  ※ 자동 임의컷 없음 — 표 보고 수동판단.
def vec(bit, key): return [PROX_PTQ[bit][L][key] for L in layers]
def recvec(bit, t): return [REC[bit][L][t] for L in layers]

print(f"{'bit':>3} | {'dtHd→conv raw/partN':>20} | {'dtGd→conv raw/partN':>20} | {'nHd2→short partN':>16} | calib Hd/Gd | dtHd<0")
AB = {}
for bit in BITS:
    if bit not in REC: continue
    cR, sR = recvec(bit, CONV), recvec(bit, SHORT)
    dtHd, dtGd, nHd = vec(bit, 'dtHd'), vec(bit, 'dtGd'), vec(bit, 'normHd2')
    r_hd, r_hd_p = spearman(dtHd, cR), partial_spearman(dtHd, cR, Nvec)
    r_gd, r_gd_p = spearman(dtGd, cR), partial_spearman(dtGd, cR, Nvec)
    r_nh_p = partial_spearman(nHd, sR, Nvec)
    def medratio(pv):
        rr = [m / (0.5 * p) for m, p in zip(cR, pv) if p > 1e-12]
        return float(np.median(rr)) if rr else float('nan')
    nneg = sum(1 for v in dtHd if v < 0)
    AB[bit] = dict(r_hd_p=r_hd_p, r_gd_p=r_gd_p, r_nh_p=r_nh_p,
                   mr_hd=medratio(dtHd), mr_gd=medratio(dtGd), dtHd_neg=nneg)
    print(f"W{bit:>2} | {r_hd:>8.2f}/{r_hd_p:>8.2f} | {r_gd:>8.2f}/{r_gd_p:>8.2f} | {r_nh_p:>15.2f} | {AB[bit]['mr_hd']:>4.0f}/{AB[bit]['mr_gd']:>4.0f} | {nneg}/{len(layers)}")

print('\n[A/B 해석 — 표를 보고 *수동* 판단 (자동컷 없음)]')
print('  A 경향(추정기 탓): dtGd→conv partN 이 dtHd보다 *뚜렷이* 높고 + calib Gd배율↓(1쪽) + dtHd<0 층 많음')
print('  B 경향(본질=보상): dtGd로도 conv partN 여전히 낮음 → damage≠recovery 음성결과 + 경험적 역전에 집중')
print(f'  (3 seed 예비 — 부호·대소만. 5 seed서 확정. 본무대 W{MAIN_BIT} 위주.)')

## S1.5b — 3-anchor(FP vs PTQ) + cos(Δ,δ)  *(보조 증거)*

de-quant(δ 되돌리기) vs compensation(다른 최적해). fixed-scale라 *완벽판정은 불가* → 보조로만.

In [ ]:
# S1.5b-1: 3-anchor — FP vs PTQ 곡률 중 어느 게 회복 예측? (같은 δ)
bit = MAIN_BIT
pm = make_ptq_model(fp_model, bit); deltas = quant_error(pm)
A_ptq = PROX_PTQ[bit]
A_fp  = anchor_proxies(fp_model, deltas, calib_loader, layers, n_batches=N_BATCH_PROXY)
_json.dump(A_fp, open(f'{OUTDIR}/s1_5_anchor_fp_w{bit}.json', 'w'))
cR, sR = recvec(bit, CONV), recvec(bit, SHORT)
for key, tgt, tname in [('normHd2', sR, 'short'), ('dtGd', cR, 'conv')]:
    rp_ptq = partial_spearman([A_ptq[L][key] for L in layers], tgt, Nvec)
    rp_fp  = partial_spearman([A_fp[L][key]  for L in layers], tgt, Nvec)
    print(f'{key}→{tname}:  PTQ-anchor partN {rp_ptq:+.2f}  |  FP-anchor partN {rp_fp:+.2f}')
print('예측: PTQ 지점(g≈Hδ)이 회복신호↑, FP 지점↓ → "회복은 FP32 원점복귀 아님" 방증(보조)')

In [ ]:
# S1.5b-2: cos(Δ,δ) — QAT 이동이 de-quant 방향과 *얼마나 정렬되나* (보조 진단)
# ⚠ fixed-scale라 effective가 grid에 묶임 → "FP32 회귀?"를 완벽판정 못함. cos 낮다고 "무조건 다른 최적점"까진 과함.
bit = MAIN_BIT
order = sorted(layers, key=lambda L: REC[bit][L][CONV], reverse=True)
dir_layers = order[:6]
scales = compute_scales(fp_model, bit)
fp_w = {L: dict(fp_model.named_modules())[L].weight.detach().clone() for L in dir_layers}
print(f'{"layer":>16} | cos_lat | cos_eff | proj | orth')
DIRS = {}
for L in dir_layers:
    res, state = recover_subset(fp_model, bit, [L], train_loader, val_loader,
                                steps=DIR_STEPS, eval_at=(DIR_STEPS,), seed=0, return_state=True)
    d = recovery_direction(fp_w[L], state[L + '.weight'], scales[L], bit)
    DIRS[L] = d
    print(f'{L:>16} | {d["cos_latent"]:+.3f} | {d["cos_eff"]:+.3f} | {d["proj_ratio"]:.2f} | {d["orth_ratio"]:.2f}')
_json.dump(DIRS, open(f'{OUTDIR}/s1_5_direction_w{bit}.json', 'w'))
mc = float(np.nanmean([DIRS[L]['cos_eff'] for L in dir_layers]))
print(f'\n평균 cos_eff = {mc:+.3f}  → 높으면 de-quant 정렬↑. 낮아도 "다른 최적점"은 *보조* 해석(S1.5a·anchor와 함께).')

## S1.5c — path-integrated early gradient

정적 1-shot proxy가 못 잡는 budget 의존을 동적 proxy가 잡나 + 곡률응답 family와 중복?

In [ ]:
# S1.5c: path-integrated Σ‖g(t)‖² (동적) → 단기회복 예측 + normHd2 중복 점검
bit = MAIN_BIT
PIG = {L: float(np.mean([path_integrated_grad(fp_model, bit, L, train_loader, T=PIG_T, seed=s)
                         for s in SEEDS])) for L in layers}
_json.dump(PIG, open(f'{OUTDIR}/s1_5_pathint_w{bit}.json', 'w'))
sR  = recvec(bit, SHORT)
pig = [PIG[L] for L in layers]; nHd = [PROX_PTQ[bit][L]['normHd2'] for L in layers]
print(f'path-int → short  partN = {partial_spearman(pig, sR, Nvec):+.2f}')
print(f'normHd2  → short  partN = {partial_spearman(nHd, sR, Nvec):+.2f}  (기존, 비교)')
print(f'path-int vs normHd2 상관 = {spearman(pig, nHd):+.2f}  (높으면 중복=곡률응답 family)')

## S1.5d — 가산성 (S2 게이트)

단일층 회복 점수가 subset 회복으로 전이되나. `I_ij = R({i,j}) − R(i) − R(j)`.

In [ ]:
# S1.5d: 가산성 I_ij (상위 ADD_TOPK 층, 단기예산)
bit = MAIN_BIT
top = sorted(layers, key=lambda L: REC[bit][L][SHORT], reverse=True)[:ADD_TOPK]
print('상위', ADD_TOPK, '층:', top)
pm = make_ptq_model(fp_model, bit); _, pl0 = evaluate_full(pm, val_loader)
def R_of(subset):
    r = recover_subset(fp_model, bit, subset, train_loader, val_loader, steps=SHORT, eval_at=(SHORT,), seed=0)
    return pl0 - r[SHORT]['loss']
single = {L: R_of([L]) for L in top}
I = {}
for i, j in itertools.combinations(top, 2):
    I[(i, j)] = R_of([i, j]) - single[i] - single[j]
med_abs_I  = float(np.median([abs(v) for v in I.values()]))
med_single = float(np.median(list(single.values())))
ratio = med_abs_I / abs(med_single) if med_single else float('nan')
print(f'단일 R 중앙값 {med_single:.4f} | |I_ij| 중앙값 {med_abs_I:.4f} | 비율 {ratio:.2f}')
print('대략 기준(엄밀X): 비율 작음(~0.2 미만)→ 가법적, top-k 정당 | 큼→ 상호작용=발견, greedy 필요')
_json.dump({f'{i}|{j}': v for (i, j), v in I.items()}, open(f'{OUTDIR}/s1_5_additivity_w{bit}.json', 'w'))

## 요약 — A/B 판정 + 메커니즘 + 가산성

In [ ]:
# 요약 저장 — s1_5_summary.json
summary = dict(
    main_bit=MAIN_BIT, seeds=SEEDS,
    s1_5a_AB={str(b): AB[b] for b in AB},
    s1_5b_mean_cos_eff=float(np.nanmean([DIRS[L]['cos_eff'] for L in DIRS])) if DIRS else None,
    s1_5c_pathint_short_partN=partial_spearman([PIG[L] for L in layers], recvec(MAIN_BIT, SHORT), Nvec),
    s1_5d_med_abs_I=med_abs_I, s1_5d_med_single=med_single, s1_5d_ratio=ratio)
_json.dump(summary, open(f'{OUTDIR}/s1_5_summary.json', 'w'), indent=2, ensure_ascii=False)
print('저장 s1_5_summary.json')
print(_json.dumps(summary, indent=2, ensure_ascii=False))